In [ ]:
import pandas as pd
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load test data
BASE = "/content/drive/MyDrive/CS685/linkedin"
test_df = pd.read_csv(f"{BASE}/ner_data_v3/bio_test.csv")

# Check how it looks
print(f"Total test sentences: {len(test_df['sent_id'].unique())}")

# Load the original annotations to get gold spans
annot_df = pd.read_csv(f"{BASE}/sentences_annotated_clean.csv")

# Get only test sentences
test_sent_ids = test_df['sent_id'].unique()
test_with_gold = annot_df[annot_df['sent_id'].isin(test_sent_ids)].copy()

# Parse gold spans
def parse_spans(raw):
    if pd.isna(raw):
        return []
    return [s.strip() for s in str(raw).split(';') if s.strip()]

test_with_gold['gold_spans_list'] = test_with_gold['spans'].apply(parse_spans)

# Filter to only sentences WITH skills
test_with_skills = test_with_gold[test_with_gold['gold_spans_list'].apply(len) > 0].copy()

print(f"Test sentences with skills: {len(test_with_skills)}")

# Explode so each skill mention gets its own row
rows = []
for _, row in test_with_skills.iterrows():
    for skill_text in row['gold_spans_list']:
        rows.append({
            'sent_id': row['sent_id'],
            'sentence': row['sentence'],
            'skill_mention': skill_text,
            'esco_id': '',  # TO BE FILLED
            'esco_label': '',  # TO BE FILLED
            'notes': ''
        })

annotation_df = pd.DataFrame(rows)
print(f"Total skill mentions to annotate: {len(annotation_df)}")

# Save for annotation
annotation_df.to_csv(f"{BASE}/test_skills_for_esco_annotation.csv", index=False)
print("Saved! Now annotate this file.")

Total test sentences: 105
Test sentences with skills: 64
Total skill mentions to annotate: 190
Saved! Now annotate this file.


In [ ]:
# Load your annotations
annotated = pd.read_csv(f"{BASE}/test_skills_annotated.csv")

print(f"Total annotations: {len(annotated)}")
print(f"NIL annotations: {(annotated['esco_id'] == 'NIL').sum()}")

# Clean up ESCO IDs - extract just the ID part
def extract_esco_id(uri):
    if pd.isna(uri) or uri == 'NIL':
        return 'NIL'
    # Extract the ID from URI
    # "http://data.europa.eu/esco/skill/abc123" → "abc123"
    return uri.split('/')[-1]

annotated['esco_id_clean'] = annotated['esco_id'].apply(extract_esco_id)

# Save clean version
annotated.to_csv(f"{BASE}/test_gold_esco.csv", index=False)
print("Gold ESCO annotations ready!")

Total annotations: 190
NIL annotations: 57
Gold ESCO annotations ready!
